# Clustering

It is an unsupervised machine learning technique used to group similar data points into clusters without having predefined target labels.

The goal is to make:

- Data points within the same cluster as similar as possible.
- Data points from different clusters as different as possible

## Workflow

1. Load dataset
2. EDA
3. Preprocessing
4. Select features
5. Scale features
6. Train clustering model
7. Predict / assign clusters
8. Evaluate clusters
9. Visualize
10. Compare models
11. Improve

Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("data.csv")

df.head()

EDA

In [ ]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

In [ ]:
df.describe()

In [ ]:
print("\nData Types:")
print(df.dtypes)

### Preprocessing

Separate Features

In [ ]:
X = df.select_dtypes(include=np.number).copy()

print("Features:")
print(X.columns.tolist())

Handle missing values

In [ ]:
X = X.dropna()

print("Shape after missing-value handling:", X.shape)

Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled[:5]

### Evaluation

Useful metrics:

- **Silhouette Score:** higher is generally better
- **Davies-Bouldin Score:** lower is better
- **Calinski-Harabasz Score:** higher is better

In [ ]:
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

def evaluate_clustering(name, X_data, labels):
    # Metrics require at least 2 clusters.
    n_clusters = len(np.unique(labels))

    if n_clusters < 2:
        return {
            "Model": name,
            "Clusters": n_clusters,
            "Silhouette": np.nan,
            "Davies-Bouldin": np.nan,
            "Calinski-Harabasz": np.nan
        }

    return {
        "Model": name,
        "Clusters": n_clusters,
        "Silhouette": silhouette_score(X_data, labels),
        "Davies-Bouldin": davies_bouldin_score(X_data, labels),
        "Calinski-Harabasz": calinski_harabasz_score(X_data, labels)
    }

# K-Means

**Use:** When you expect compact, roughly spherical groups and you can choose the number of clusters `k`.

Main parameter:
- `n_clusters` → number of groups

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

kmeans_labels = kmeans.fit_predict(X_scaled)

result_kmeans = evaluate_clustering(
    "K-Means",
    X_scaled,
    kmeans_labels
)

result_kmeans

## Choosing K with the Elbow Method

Try several values of `k` and inspect the inertia curve.

In [ ]:
inertias = []

k_values = range(2, 11)

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_values, inertias, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

# Hierarchical Clustering

**Use:** When you want a hierarchy of clusters and a dendrogram can help you understand the grouping structure.

Main parameters:
- `n_clusters`
- `linkage`

In [ ]:
from sklearn.cluster import AgglomerativeClustering

hierarchical = AgglomerativeClustering(
    n_clusters=3,
    linkage="ward"
)

hierarchical_labels = hierarchical.fit_predict(X_scaled)

result_hierarchical = evaluate_clustering(
    "Hierarchical Clustering",
    X_scaled,
    hierarchical_labels
)

result_hierarchical

### Dendrogram

A dendrogram helps visualize hierarchical merging. It can help choose a useful number of clusters.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(10, 6))
dendrogram(Z, truncate_mode="level", p=5)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Samples / merged groups")
plt.ylabel("Distance")
plt.show()

# DBSCAN

**Use:** When clusters may have irregular shapes and the data contains noise/outliers.

Main parameters:
- `eps` → neighborhood radius
- `min_samples` → minimum points needed for a dense region

DBSCAN can label noise as `-1`.

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(
    eps=0.5,
    min_samples=5
)

dbscan_labels = dbscan.fit_predict(X_scaled)

result_dbscan = evaluate_clustering(
    "DBSCAN",
    X_scaled,
    dbscan_labels
)

result_dbscan

# Gaussian Mixture Model (GMM)

**Use:** When you want probabilistic/soft cluster membership and clusters can be represented by Gaussian distributions.

Main parameter:
- `n_components` → number of mixture components

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(
    n_components=3,
    random_state=42
)

gmm_labels = gmm.fit_predict(X_scaled)

result_gmm = evaluate_clustering(
    "Gaussian Mixture",
    X_scaled,
    gmm_labels
)

result_gmm

# Mean Shift

**Use:** When you want the algorithm to estimate cluster centers from data density instead of specifying the number of clusters directly.

It can be more computationally expensive on large datasets.

In [ ]:
from sklearn.cluster import MeanShift

mean_shift = MeanShift()

mean_shift_labels = mean_shift.fit_predict(X_scaled)

result_mean_shift = evaluate_clustering(
    "Mean Shift",
    X_scaled,
    mean_shift_labels
)

result_mean_shift

# Model Comparison

- **Silhouette:** higher is better
- **Davies-Bouldin:** lower is better
- **Calinski-Harabasz:** higher is better

In [ ]:
results = pd.DataFrame([
    result_kmeans,
    result_hierarchical,
    result_dbscan,
    result_gmm,
    result_mean_shift
])

results.sort_values(
    "Silhouette",
    ascending=False
).reset_index(drop=True)

In [ ]:
comparison = results.sort_values("Silhouette")

plt.figure(figsize=(10, 6))
plt.barh(comparison["Model"], comparison["Silhouette"])
plt.xlabel("Silhouette Score (higher is better)")
plt.title("Clustering Model Comparison")
plt.show()

## Cluster Visualization

For visualization, reduce the features to 2 dimensions. PCA is useful for a simple visual check.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)

X_2d = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=kmeans_labels)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-Means Clusters")
plt.show()

# Model Improvement

**Baseline → Choose useful features → Scale → Try different cluster counts/parameters → Compare metrics → Visualize → Domain interpretation → Final clustering**



### For K-Means
Tune:
- `n_clusters`

Use:
- Elbow method
- Silhouette score

### For DBSCAN
Tune:
- `eps`
- `min_samples`

### For Hierarchical
Tune:
- `n_clusters`
- `linkage`

### For GMM
Tune:
- `n_components`
- covariance type

# Notes

**K-Means** → compact groups; choose `k`

**Hierarchical** → hierarchy of groups; use dendrogram

**DBSCAN** → density-based groups + noise; irregular shapes

**GMM** → probabilistic/soft clustering

**Mean Shift** → density-based cluster centers; number of clusters can emerge from the data

### Key difference

K-Means → distance to cluster centers

Hierarchical → progressively merge/split groups

DBSCAN → density of nearby points

GMM → probability of belonging to Gaussian components

Mean Shift → move toward dense regions